In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [2]:
# -------------------------------------------------
# 1 Load datasets
# -------------------------------------------------

male = pd.read_excel("male_file.xlsx")
female = pd.read_excel("female_file.xlsx")

data = pd.concat([male, female], ignore_index=True)

In [3]:
# -------------------------------------------------
# 2 Encode Sex column
# -------------------------------------------------

data["Sex"] = data["Sex"].map({"M":0, "F":1})

In [4]:
# -------------------------------------------------
# 3 Define features / labels / groups
# -------------------------------------------------

target = "Sex"
group_column = "Tree_ID"

X = data.drop(columns=[target, group_column])
y = data[target]
groups = data[group_column]

In [5]:
# -------------------------------------------------
# 4 Define ML models
# -------------------------------------------------

models = {

"RandomForest": RandomForestClassifier(
    n_estimators=300,
    random_state=42
),

"SVM": SVC(
    kernel="rbf",
    probability=True
),

"LogisticRegression": LogisticRegression(
    max_iter=2000
),

"GradientBoosting": GradientBoostingClassifier()
}

In [6]:
# -------------------------------------------------
# 5 Group Cross Validation
# -------------------------------------------------

gkf = GroupKFold(n_splits=5)

results = []


for name, model in models.items():

    acc_scores = []
    f1_scores = []
    prec_scores = []
    rec_scores = []

    for train_idx, test_idx in gkf.split(X, y, groups):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]


        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])


        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)


        acc_scores.append(accuracy_score(y_test, y_pred))
        f1_scores.append(f1_score(y_test, y_pred))
        prec_scores.append(precision_score(y_test, y_pred))
        rec_scores.append(recall_score(y_test, y_pred))


    results.append({

        "Model": name,
        "Accuracy": np.mean(acc_scores),
        "F1-score": np.mean(f1_scores),
        "Precision": np.mean(prec_scores),
        "Recall": np.mean(rec_scores)

    })


results_df = pd.DataFrame(results)

print("\nModel Comparison:\n")
print(results_df)


Model Comparison:

                Model  Accuracy  F1-score  Precision    Recall
0        RandomForest  0.489306  0.441080   0.472461  0.425339
1                 SVM  0.507229  0.380509   0.495732  0.328225
2  LogisticRegression  0.493506  0.407736   0.486565  0.376197
3    GradientBoosting  0.496652  0.436392   0.479759  0.407510
